# Homework 27: Causal Interventions and Feature Steering

**Audience.** Students who have trained an SAE on PicoGPT residual activations.

**Prerequisites.** SAE encoding and decoder directions; next-token probabilities; autoregressive sampling; mean and frequency statistics.

**Learning goals.** By the end, you will be able to:

- inspect the contexts that activate an SAE feature;
- ablate or amplify one feature direction in the residual stream;
- compare SAE steering with a matched random direction;
- compare internal steering with direct output-logit manipulation;
- measure both target behavior and off-target model degradation.

Feature steering is a causal experiment, not merely a way to produce an entertaining sample. Every claim should name an intervention, a control, and a measurable outcome.


## Outline

1. Load the fixed GPT and SAE
2. Select and inspect a held-out period-enriched feature
3. Define ablation, amplification, and random controls
4. Measure next-token effects
5. Generate with fixed seeds and compute sentence statistics
6. Notebook checkpoints


In [1]:
# S1: Imports, paths, and the fixed PicoGPT checkpoint
from pathlib import Path
import re
from statistics import fmean
import sys

import torch
from torch import nn

if Path("pico_gpt.py").exists():
    COURSE_DIRECTORY = Path(".")
elif Path("Homework2026/pico_gpt.py").exists():
    COURSE_DIRECTORY = Path("Homework2026")
else:
    raise FileNotFoundError("Place pico_gpt.py beside this notebook.")
sys.path.insert(0, str(COURSE_DIRECTORY.resolve()))

from pico_gpt import (
    build_token_stream,
    get_batch,
    load_checkpoint,
    make_demo_stories,
    split_stories,
)

_ = torch.manual_seed(158)
reference_pairs = [
    (
        "TinyStories",
        COURSE_DIRECTORY / "pico_gpt_tinystories_reference.pt",
        COURSE_DIRECTORY / "pico_gpt_tinystories_reference_sae.pt",
    ),
    (
        "fast demonstration",
        COURSE_DIRECTORY / "pico_gpt_reference.pt",
        COURSE_DIRECTORY / "pico_gpt_reference_sae.pt",
    ),
]
available_pair = next(
    (
        (name, gpt_candidate, sae_candidate)
        for name, gpt_candidate, sae_candidate in reference_pairs
        if gpt_candidate.exists() and sae_candidate.exists()
    ),
    None,
)
if available_pair is None:
    raise FileNotFoundError(
        "Matching immutable GPT and SAE reference checkpoints are missing."
    )

reference_kind, gpt_path, sae_path = available_pair
model, tokenizer, gpt_metadata = load_checkpoint(gpt_path)

model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
print("reference pair:", reference_kind)


reference pair: fast demonstration


This notebook loads an immutable, matching GPT/SAE pair. Student training outputs use different filenames, so rerunning an earlier notebook cannot change feature identities or graded values. When the instructor-trained TinyStories pair is present, it is preferred automatically; otherwise the included fast pair demonstrates the same machinery.


In [2]:
# S2: The same SAE architecture used in Homework 26
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dimension, number_of_features):
        super().__init__()
        self.input_dimension = input_dimension
        self.number_of_features = number_of_features
        self.encoder = nn.Linear(input_dimension, number_of_features)
        self.decoder_directions = nn.Parameter(
            torch.empty(number_of_features, input_dimension)
        )
        self.decoder_bias = nn.Parameter(torch.zeros(input_dimension))
        nn.init.kaiming_uniform_(self.decoder_directions)
        self.normalize_decoder()
        with torch.no_grad():
            self.encoder.weight.copy_(self.decoder_directions)
            self.encoder.bias.zero_()

    def encode(self, activations):
        return torch.relu(self.encoder(activations - self.decoder_bias))

    def forward(self, activations):
        features = self.encode(activations)
        reconstruction = features @ self.decoder_directions + self.decoder_bias
        return reconstruction, features

    @torch.no_grad()
    def normalize_decoder(self):
        norms = self.decoder_directions.norm(dim=1, keepdim=True).clamp_min(1e-8)
        self.decoder_directions.div_(norms)


In [3]:
# S3: Load the matching SAE, then create calibration and final-test data
state = torch.load(sae_path, map_location="cpu", weights_only=True)
target_layer = int(state["target_layer"])
sae = SparseAutoencoder(
    int(state["input_dimension"]), int(state["number_of_features"])
)
sae.load_state_dict(state["state_dict"])
sae.eval()

stories = make_demo_stories()
train_stories, validation_stories = split_stories(stories)
calibration_stories, final_test_stories = split_stories(
    validation_stories, validation_fraction=0.5, seed=277
)
calibration_stream = build_token_stream(calibration_stories, tokenizer)
final_test_stream = build_token_stream(final_test_stories, tokenizer)
period_id = tokenizer.stoi["."]

@torch.no_grad()
def collect_with_context(stream, number_of_batches, seed):
    generator = torch.Generator().manual_seed(seed)
    activations, labels, input_batches, target_batches = [], [], [], []
    for _ in range(number_of_batches):
        inputs, targets = get_batch(
            stream, 8, model.config.block_size, generator
        )
        _, _, cache = model(inputs, return_cache=True)
        residual = cache["blocks"][target_layer]["residual_post"]
        activations.append(residual.reshape(-1, model.config.d_model))
        labels.append((targets == period_id).reshape(-1))
        input_batches.append(inputs)
        target_batches.append(targets)
    return (
        torch.cat(activations),
        torch.cat(labels),
        torch.cat(input_batches),
        torch.cat(target_batches),
    )

(
    calibration_activations,
    calibration_period_labels,
    calibration_inputs,
    calibration_targets,
) = collect_with_context(calibration_stream, 10, 271)
(
    final_test_activations,
    final_test_period_labels,
    final_test_inputs,
    final_test_targets,
) = collect_with_context(final_test_stream, 10, 272)

assert set(calibration_stories).isdisjoint(final_test_stories)
print("SAE checkpoint:", sae_path.name)
print("SAE features:", sae.number_of_features)
print("target layer:", target_layer)
print(
    "calibration/test activations:",
    tuple(calibration_activations.shape),
    tuple(final_test_activations.shape),
)


SAE checkpoint: pico_gpt_reference_sae.pt
SAE features: 256
target layer: 0
calibration/test activations: (3840, 64) (3840, 64)


## 1. Select and inspect a feature

We select the feature on a calibration split rather than trusting a feature number saved during training. The selection label is “the next token is a period.” The final-test split remains untouched until the causal evaluation.


In [4]:
# S4: Standardized calibration enrichment and unique top contexts
with torch.no_grad():
    calibration_features = sae.encode(calibration_activations)
    period_means = calibration_features[
        calibration_period_labels
    ].mean(dim=0)
    other_means = calibration_features[
        ~calibration_period_labels
    ].mean(dim=0)
    feature_scales = calibration_features.std(dim=0).clamp_min(1e-6)
    enrichment = (period_means - other_means) / feature_scales
    selected_feature = int(enrichment.argmax())
    selected_activations = calibration_features[:, selected_feature]

top_flat_indices = torch.topk(selected_activations, k=50).indices
sequence_length = model.config.block_size
top_contexts = []
seen_contexts = set()
for flat_index in top_flat_indices:
    row = int(flat_index) // sequence_length
    position = int(flat_index) % sequence_length
    start = max(0, position - 6)
    context = tokenizer.decode(
        calibration_inputs[row, start : position + 1]
    )
    next_token = tokenizer.itos[int(calibration_targets[row, position])]
    labeled_context = f"{context}  →  next token: {next_token}"
    if labeled_context not in seen_contexts:
        top_contexts.append(labeled_context)
        seen_contexts.add(labeled_context)
    if len(top_contexts) == 8:
        break

print("selected feature:", selected_feature)
print("period enrichment:", round(float(enrichment[selected_feature]), 5))
for rank, context in enumerate(top_contexts, start=1):
    print(f"{rank}. ...{context}")


selected feature: 6
period enrichment: 2.60742
1. ...Ava wanted to visit the river  →  next token: .
2. ...lost a red key at the river  →  next token: .
3. ...Rain fell over the park  →  next token: .
4. ...happy  →  next token: .
5. ...Rain fell over the garden  →  next token: .
6. ...lost a purple hat at the school  →  next token: .
7. ...Cora. They played beside the school  →  next token: .
8. ...lost a gold book at the garden  →  next token: .


Top contexts suggest a hypothesis; they do not prove what the feature does. We now intervene and compare with controls.


In [5]:
# S5: Feature ablation, true coefficient amplification, and random controls
feature_direction = sae.decoder_directions[selected_feature].detach()
random_generator = torch.Generator().manual_seed(273)
random_directions = torch.randn(
    20,
    feature_direction.numel(),
    generator=random_generator,
)
random_directions = random_directions / random_directions.norm(
    dim=1, keepdim=True
)
random_direction = random_directions[0]

def selected_feature_values(residual):
    return sae.encode(residual)[..., selected_feature]

def ablate_feature(layer_index, residual):
    if layer_index != target_layer:
        return residual
    feature_values = selected_feature_values(residual)
    return residual - feature_values.unsqueeze(-1) * feature_direction

def amplify_feature(layer_index, residual, factor=2.0):
    if layer_index != target_layer:
        return residual
    feature_values = selected_feature_values(residual)
    extra = (factor - 1.0) * feature_values.unsqueeze(-1)
    return residual + extra * feature_direction

def make_random_control(direction):
    def random_control(layer_index, residual):
        if layer_index != target_layer:
            return residual
        feature_values = selected_feature_values(residual)
        return residual + feature_values.unsqueeze(-1) * direction
    return random_control

add_random_direction = make_random_control(random_direction)

def zero_strength_control(layer_index, residual):
    if layer_index == target_layer:
        return residual + 0.0 * feature_direction
    return residual


## 2. A fixed next-token experiment

First measure one prompt without sampling. Directly subtracting from the period logit is a necessary control: it trivially lowers the period probability. SAE steering is interesting only if an internal feature has a broader, structured effect.


In [6]:
# S6: Choose a feature-active, multi-token calibration prompt
with torch.no_grad():
    calibration_logits, _ = model(calibration_inputs)
    calibration_ablated_logits, _ = model(
        calibration_inputs, intervention=ablate_feature
    )
    calibration_amplified_logits, _ = model(
        calibration_inputs, intervention=amplify_feature
    )
    calibration_period_probabilities = torch.softmax(
        calibration_logits, dim=-1
    )[:, :, period_id].reshape(-1)
    calibration_ablated_probabilities = torch.softmax(
        calibration_ablated_logits, dim=-1
    )[:, :, period_id].reshape(-1)
    calibration_amplified_probabilities = torch.softmax(
        calibration_amplified_logits, dim=-1
    )[:, :, period_id].reshape(-1)

flat_positions = torch.arange(selected_activations.numel())
candidate_mask = (
    (selected_activations > 1e-4)
    & (flat_positions % model.config.block_size >= 2)
    & (calibration_period_probabilities > 0.05)
    & (calibration_period_probabilities < 0.95)
)
active_candidates = torch.where(candidate_mask)[0]
if len(active_candidates) == 0:
    active_candidates = torch.tensor([int(selected_activations.argmax())])
candidate_scores = (
    calibration_amplified_probabilities[active_candidates]
    - calibration_ablated_probabilities[active_candidates]
)
prompt_flat_index = int(active_candidates[candidate_scores.argmax()])
prompt_row = prompt_flat_index // model.config.block_size
prompt_position = prompt_flat_index % model.config.block_size
prompt = calibration_inputs[
    prompt_row : prompt_row + 1, : prompt_position + 1
]
prompt_text = tokenizer.decode(prompt[0])
prompt_feature_activation = float(
    selected_activations[prompt_flat_index]
)

def period_probability(logits):
    return torch.softmax(logits[0, -1], dim=-1)[period_id]

with torch.no_grad():
    baseline_logits, _ = model(prompt)
    zero_logits, _ = model(prompt, intervention=zero_strength_control)
    ablated_logits, _ = model(prompt, intervention=ablate_feature)
    amplified_logits, _ = model(prompt, intervention=amplify_feature)
    random_logits, _ = model(prompt, intervention=add_random_direction)

    direct_logits = baseline_logits.clone()
    direct_logits[0, -1, period_id] -= 4.0

next_token_results = {
    "baseline": float(period_probability(baseline_logits)),
    "feature ablation": float(period_probability(ablated_logits)),
    "feature amplification": float(period_probability(amplified_logits)),
    "matched random direction": float(period_probability(random_logits)),
    "direct period-logit suppression": float(period_probability(direct_logits)),
}

for condition, probability in next_token_results.items():
    print(f"{condition:31s} P(period)={probability:.5f}")
print("selected prompt:", prompt_text)
print("selected feature activation:", round(prompt_feature_activation, 4))

assert torch.allclose(baseline_logits, zero_logits, atol=1e-7)
assert next_token_results["direct period-logit suppression"] < next_token_results["baseline"]


baseline                        P(period)=0.87738
feature ablation                P(period)=0.42043
feature amplification           P(period)=0.96286
matched random direction        P(period)=0.93971
direct period-logit suppression P(period)=0.11587
selected prompt: carried the boat
selected feature activation: 0.8254


One prompt is only an illustration. The actual causal evaluation uses the untouched final-test split. It measures the change in the period logit at positions whose true next token is a period and at all other positions. We compare the SAE feature with twenty independently drawn unit-norm random directions.


In [7]:
# S7: Aggregate causal effects on the untouched final-test split
with torch.no_grad():
    baseline_test_logits, _ = model(final_test_inputs)

@torch.no_grad()
def period_logit_changes(intervention):
    changed_logits, _ = model(
        final_test_inputs, intervention=intervention
    )
    delta = (
        changed_logits[:, :, period_id]
        - baseline_test_logits[:, :, period_id]
    )
    return {
        "before_period": float(delta[final_test_period_labels.view_as(delta)].mean()),
        "other_positions": float(delta[~final_test_period_labels.view_as(delta)].mean()),
    }

aggregate_effects = {
    "feature ablation": period_logit_changes(ablate_feature),
    "feature amplification": period_logit_changes(amplify_feature),
}
random_effects = [
    period_logit_changes(make_random_control(direction))
    for direction in random_directions
]
random_before_period = torch.tensor([
    result["before_period"] for result in random_effects
])
aggregate_effects["random controls"] = {
    "mean_before_period": float(random_before_period.mean()),
    "std_before_period": float(random_before_period.std()),
}

for condition, result in aggregate_effects.items():
    print(condition, result)


feature ablation {'before_period': -0.4741266369819641, 'other_positions': -0.03072970360517502}
feature amplification {'before_period': 0.18273402750492096, 'other_positions': 0.03906195983290672}
random controls {'mean_before_period': -0.10900537669658661, 'std_before_period': 0.06133458390831947}


A target effect is not enough: an intervention might change punctuation simply by damaging every prediction. We therefore compare language-model cross-entropy on the same fixed final-test examples. Lower is better, and the increase above baseline is a quality cost.


In [8]:
# S8: Measure predictive-quality cost on final-test examples
quality_inputs = final_test_inputs[:16]
quality_targets = final_test_targets[:16]

with torch.no_grad():
    baseline_quality_logits, baseline_quality_loss = model(
        quality_inputs, quality_targets
    )
    _, ablation_quality_loss = model(
        quality_inputs, quality_targets, intervention=ablate_feature
    )
    _, amplification_quality_loss = model(
        quality_inputs, quality_targets, intervention=amplify_feature
    )
    _, random_quality_loss = model(
        quality_inputs, quality_targets, intervention=add_random_direction
    )

    direct_quality_logits = baseline_quality_logits.clone()
    direct_quality_logits[:, :, period_id] -= 4.0
    direct_quality_loss = nn.functional.cross_entropy(
        direct_quality_logits.reshape(-1, direct_quality_logits.shape[-1]),
        quality_targets.reshape(-1),
    )

quality_losses = {
    "baseline": float(baseline_quality_loss),
    "feature ablation": float(ablation_quality_loss),
    "feature amplification": float(amplification_quality_loss),
    "matched random direction": float(random_quality_loss),
    "direct period-logit suppression": float(direct_quality_loss),
}
for condition, loss in quality_losses.items():
    print(f"{condition:31s} validation loss={loss:.4f}")


baseline                        validation loss=0.6407
feature ablation                validation loss=0.6422
feature amplification           validation loss=0.6437
matched random direction        validation loss=0.6416
direct period-logit suppression validation loss=0.6969


## 3. Controlled generation

Each condition uses the same prompt, temperature, top-k cutoff, and random seed. The period-logit bias is applied to raw logits before temperature scaling, so `-4.0` means the same intervention in the probability and generation experiments.


In [9]:
# S9: Generation with both residual interventions and an optional output-logit bias
@torch.no_grad()
def controlled_generate(
    prompt,
    max_new_tokens,
    intervention=None,
    period_logit_bias=0.0,
    seed=1,
    temperature=0.8,
    top_k=8,
):
    generated = prompt.clone()
    generator = torch.Generator().manual_seed(seed)

    for _ in range(max_new_tokens):
        context = generated[:, -model.config.block_size:]
        logits, _ = model(context, intervention=intervention)
        next_logits = logits[:, -1].clone()
        next_logits[:, period_id] += period_logit_bias
        next_logits = next_logits / temperature

        threshold = torch.topk(next_logits, k=top_k).values[:, [-1]]
        next_logits[next_logits < threshold] = float("-inf")
        probabilities = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probabilities, 1, generator=generator)
        generated = torch.cat([generated, next_id], dim=1)
        if bool(torch.all(next_id == tokenizer.eos_id)):
            break
    return generated

conditions = {
    "baseline": {"intervention": None, "period_logit_bias": 0.0},
    "feature ablation": {"intervention": ablate_feature, "period_logit_bias": 0.0},
    "feature amplification": {"intervention": amplify_feature, "period_logit_bias": 0.0},
    "random direction": {"intervention": add_random_direction, "period_logit_bias": 0.0},
    "direct period suppression": {"intervention": None, "period_logit_bias": -4.0},
}

generated_by_condition = {
    name: controlled_generate(prompt, max_new_tokens=60, **arguments)
    for name, arguments in conditions.items()
}

for name, ids in generated_by_condition.items():
    print(f"\n{name.upper()}\n{tokenizer.decode(ids[0])}")



BASELINE
carried the boat. "What a good day!" Ella said. Ella cheered.

FEATURE ABLATION
carried the boat with Ben. They played beside the garden. Both friends were happy.

FEATURE AMPLIFICATION
carried the boat. "What a good day!" Ella said. Ella cheered.

RANDOM DIRECTION
carried the boat. "What a good day!" Ella said. Ella cheered.

DIRECT PERIOD SUPPRESSION
carried the boat with Ben. They played beside the garden. Both friends were happy.


In [10]:
# S10: Objective statistics across several prompts and seeds
WORD_PATTERN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def generation_statistics(ids, prompt_length):
    generated_ids = ids[0, prompt_length:].tolist()
    tokens = [tokenizer.itos[index] for index in generated_ids]
    ordinary_tokens = [
        token for token in tokens
        if token not in {"<pad>", "<unk>", "<bos>", "<eos>"}
    ]
    number_of_words = sum(bool(WORD_PATTERN.fullmatch(token)) for token in ordinary_tokens)
    number_of_periods = ordinary_tokens.count(".")
    return {
        "generated_tokens": len(generated_ids),
        "words": number_of_words,
        "periods": number_of_periods,
        "periods_per_100_tokens": round(
            100 * number_of_periods / max(1, len(ordinary_tokens)), 2
        ),
        "words_per_period": (
            None
            if number_of_periods == 0
            else round(number_of_words / number_of_periods, 2)
        ),
    }

statistics = {
    name: generation_statistics(ids, prompt.shape[1])
    for name, ids in generated_by_condition.items()
}

static_evaluation_texts = [
    "Ava found a blue kite",
    "Soon they were warm",
]
evaluation_prompts = [prompt]
for text in static_evaluation_texts:
    ids = [tokenizer.bos_id] + tokenizer.encode(
        text, add_special_tokens=False
    )
    evaluation_prompts.append(torch.tensor([ids]))
evaluation_seeds = [274, 275, 276]
all_generation_statistics = {name: [] for name in conditions}

for evaluation_prompt in evaluation_prompts:
    for evaluation_seed in evaluation_seeds:
        for name, arguments in conditions.items():
            generated = controlled_generate(
                evaluation_prompt,
                max_new_tokens=60,
                seed=evaluation_seed,
                **arguments,
            )
            all_generation_statistics[name].append(
                generation_statistics(
                    generated, evaluation_prompt.shape[1]
                )
            )

aggregate_generation_statistics = {}
for name, records in all_generation_statistics.items():
    defined_lengths = [
        record["words_per_period"]
        for record in records
        if record["words_per_period"] is not None
    ]
    aggregate_generation_statistics[name] = {
        "runs": len(records),
        "mean_generated_tokens": round(
            fmean(record["generated_tokens"] for record in records), 2
        ),
        "mean_periods_per_100_tokens": round(
            fmean(record["periods_per_100_tokens"] for record in records),
            2,
        ),
        "mean_words_per_period": (
            None if not defined_lengths else round(fmean(defined_lengths), 2)
        ),
        "runs_with_no_period": sum(
            record["periods"] == 0 for record in records
        ),
    }

aggregate_generation_statistics


{'baseline': {'runs': 9,
  'mean_generated_tokens': 16.44,
  'mean_periods_per_100_tokens': 23.96,
  'mean_words_per_period': 2.98,
  'runs_with_no_period': 0},
 'feature ablation': {'runs': 9,
  'mean_generated_tokens': 16.33,
  'mean_periods_per_100_tokens': 23.29,
  'mean_words_per_period': 3.24,
  'runs_with_no_period': 0},
 'feature amplification': {'runs': 9,
  'mean_generated_tokens': 16.44,
  'mean_periods_per_100_tokens': 23.96,
  'mean_words_per_period': 2.98,
  'runs_with_no_period': 0},
 'random direction': {'runs': 9,
  'mean_generated_tokens': 16.44,
  'mean_periods_per_100_tokens': 23.96,
  'mean_words_per_period': 2.98,
  'runs_with_no_period': 0},
 'direct period suppression': {'runs': 9,
  'mean_generated_tokens': 17.11,
  'mean_periods_per_100_tokens': 22.06,
  'mean_words_per_period': 3.67,
  'runs_with_no_period': 0}}

The aggregate table summarizes nine runs per condition rather than elevating one attractive sample. On the fast templated reference model, the feature may still be noisy or polysemantic. With the canonical TinyStories pair, repeat the same controlled procedure for period/closure, dialogue, animals, or positive endings.


## Notebook checkpoints


In [11]:
# S11: Deterministic checkpoint record
checkpoint_27 = {
    "reference_kind": reference_kind,
    "gpt_checkpoint": gpt_path.name,
    "sae_checkpoint": sae_path.name,
    "target_layer": target_layer,
    "selected_feature": selected_feature,
    "feature_direction_shape": tuple(feature_direction.shape),
    "feature_direction_norm": round(float(feature_direction.norm()), 6),
    "zero_strength_matches_baseline": bool(
        torch.allclose(baseline_logits, zero_logits, atol=1e-7)
    ),
    "next_token_period_probabilities": {
        name: round(value, 6) for name, value in next_token_results.items()
    },
    "quality_losses": {
        name: round(value, 6) for name, value in quality_losses.items()
    },
    "aggregate_logit_effects": aggregate_effects,
    "top_contexts": top_contexts,
    "illustrative_generation_statistics": statistics,
    "aggregate_generation_statistics": aggregate_generation_statistics,
}
checkpoint_27


{'reference_kind': 'fast demonstration',
 'gpt_checkpoint': 'pico_gpt_reference.pt',
 'sae_checkpoint': 'pico_gpt_reference_sae.pt',
 'target_layer': 0,
 'selected_feature': 6,
 'feature_direction_shape': (64,),
 'feature_direction_norm': 1.0,
 'zero_strength_matches_baseline': True,
 'next_token_period_probabilities': {'baseline': 0.877377,
  'feature ablation': 0.420429,
  'feature amplification': 0.962862,
  'matched random direction': 0.939711,
  'direct period-logit suppression': 0.115865},
 'quality_losses': {'baseline': 0.640655,
  'feature ablation': 0.642205,
  'feature amplification': 0.643703,
  'matched random direction': 0.641599,
  'direct period-logit suppression': 0.696857},
 'aggregate_logit_effects': {'feature ablation': {'before_period': -0.4741266369819641,
   'other_positions': -0.03072970360517502},
  'feature amplification': {'before_period': 0.18273402750492096,
   'other_positions': 0.03906195983290672},
  'random controls': {'mean_before_period': -0.1090053766

## Pitfall and extension

**Pitfall.** Strong steering can increase the target statistic while destroying grammar or causing repetition. Always report a quality cost such as validation loss, repetition, or a matched-control comparison.

**Optional extension.** Repeat the experiment for capitalization. Suppressing a capitalization-associated feature may reduce uppercase letters without increasing sentence length. That makes an excellent correlation-versus-causation comparison with the period experiment.
